In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import torch

from accelerated_training_utils import JammerVecEnv, build_stft_observation_from_iq_batch
from tx_controller_tone_pulse_stft_varlen_3 import ActorCritic

@dataclass
class PPOConfig:
    rollout_steps: int = 128
    updates: int = 200
    gamma: float = 0.99
    lr: float = 3e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

# Discrete action space size consumed by tx_controller_tone_pulse_stft_varlen_3.ActorCritic
ACTION_DIM = 8


In [ ]:
def obs_to_model_obs(obs: Dict[str, torch.Tensor], jammer_sampling_freq: float, device: str) -> Dict[str, List[torch.Tensor]]:
    """Convert vectorized env IQ observation into the controller-v3 observation payload.

    The model now consumes only 2D STFT feature maps and no scalar_side tensor.
    """
    return build_stft_observation_from_iq_batch(
        iq1=obs["iq1"],
        iq2=obs["iq2"],
        iq3=obs["iq3"],
        intake_sample_rate_hz=jammer_sampling_freq,
        device=device,
    )


@torch.no_grad()
def sample_actions(model: ActorCritic, model_obs: Dict[str, List[torch.Tensor]]):
    actions, values, logp = model.get_action_value_logp(model_obs)
    return actions, values, logp


In [ ]:
def train_rl_loop(env: JammerVecEnv, cfg: PPOConfig, action_dim: int = ACTION_DIM):
    device = cfg.device
    policy = ActorCritic(action_dim=action_dim, in_ch=14, base_ch=24, max_tones=8).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.lr)

    obs = env.reset()
    for update in range(cfg.updates):
        obs_buf, act_buf, val_buf, logp_buf, rew_buf = [], [], [], [], []

        for _ in range(cfg.rollout_steps):
            model_obs = obs_to_model_obs(obs, env.jammer_sampling_freq, device=device)
            action_t, value_t, logp_t = policy.get_action_value_logp(model_obs)

            # The vectorized env accepts per-env action payloads.
            actions = [int(a.item()) for a in action_t.detach().cpu()]
            next_obs, rewards, dones, infos = env.step(actions)

            obs_buf.append(model_obs)
            act_buf.append(action_t)
            val_buf.append(value_t)
            logp_buf.append(logp_t)
            rew_buf.append(torch.as_tensor(rewards, dtype=torch.float32, device=device))

            obs = next_obs

        # Example placeholder objective using rewards and value baseline.
        returns = torch.stack(rew_buf, dim=0).mean(dim=0)
        values = torch.stack(val_buf, dim=0).mean(dim=0)
        loss = -(returns - values).mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if (update + 1) % 10 == 0:
            print(f"update={update + 1} loss={float(loss.item()):.4f}")

    return policy
